# Phase 1: Database Setup
In this notebook, we'll load the raw Olist CSV datasets, handle any necessary data cleaning/null values, and push them into a local SQLite database.

In [ ]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

# Define paths consistent with the project structure
PROJECT_ROOT = Path('.').resolve().parent
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

# Ensure processed directory exists
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_PROCESSED_DIR / 'logistics.db'

print(f"Raw Data Directory: {DATA_RAW_DIR}")
print(f"Database Path: {DB_PATH}")

In [ ]:
# Define file paths
orders_path = DATA_RAW_DIR / 'olist_orders_dataset.csv'
reviews_path = DATA_RAW_DIR / 'olist_order_reviews_dataset.csv'
customers_path = DATA_RAW_DIR / 'olist_customers_dataset.csv'

# Load datasets (Ensure they are placed in /data/raw first)
try:
    df_orders = pd.read_csv(orders_path)
    df_reviews = pd.read_csv(reviews_path)
    df_customers = pd.read_csv(customers_path)
    print("Successfully loaded all datasets.")
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    print("Please ensure the CSV files are placed in the /data/raw/ directory.")

## Handling Missing Values (Nulls)
The requirements dictate that we must handle missing values correctly.

In [ ]:
if 'df_orders' in locals():
    print("Orders missing values:\n", df_orders.isnull().sum())

    # Cast date columns to datetime objects properly so SQLite stores them in ISO format
    date_columns = [
        'order_purchase_timestamp', 
        'order_approved_at', 
        'order_delivered_carrier_date', 
        'order_delivered_customer_date', 
        'order_estimated_delivery_date'
    ]
    for col in date_columns:
        if col in df_orders.columns:
            df_orders[col] = pd.to_datetime(df_orders[col])

if 'df_reviews' in locals():
    print("\nReviews missing values:\n", df_reviews.isnull().sum())
    # For reviews, 'review_comment_message' and 'review_comment_title' often have nulls.
    if 'review_comment_message' in df_reviews.columns:
        df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('')
    if 'review_comment_title' in df_reviews.columns:
        df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('')

## Push to SQLite Database

In [ ]:
if 'df_orders' in locals() and 'df_reviews' in locals() and 'df_customers' in locals():
    # Create SQLite connection
    conn = sqlite3.connect(DB_PATH)

    # Save dataframes to SQL tables
    # We use if_exists='replace' to allow rerunning the notebook idempotently
    print("Pushing orders to database...")
    df_orders.to_sql('olist_orders', conn, if_exists='replace', index=False)

    print("Pushing reviews to database...")
    df_reviews.to_sql('olist_order_reviews', conn, if_exists='replace', index=False)

    print("Pushing customers to database...")
    df_customers.to_sql('olist_customers', conn, if_exists='replace', index=False)

    print("Successfully saved tables to logistics.db")

    # Verify creation by listing tables
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("\nTables in database:")
    for table in tables:
        print(f" - {table[0]}")

    conn.close()
    print("\nPhase 1 Complete!")
else:
    print("Dataframes were not loaded, skipping database insertion.")